In [1]:
pip install chromadb sentence_transformers -q

Note: you may need to restart the kernel to use updated packages.


#### Шаг 2. Подготовка данных: чанкинг

**Чанк** — фрагмент текста, который хранится как отдельная запись.

**Почему не загружать документ целиком?**

* LLM имеет ограничение на контекст (обычно 8K токенов),
* Длинный текст «размывает» смысл,
* Поиск по коротким фрагментам — точнее и быстрее.

In [3]:
# Пример

text = """
FastAPI — это современный веб-фреймворк для построения API на Python.
Он автоматически генерирует документацию и поддерживает асинхронность.

Alembic — это инструмент для управления миграциями базы данных в SQLAlchemy.
Он позволяет безопасно обновлять схему БД при изменении моделей.

Docker — это платформа для контейнеризации приложений.
Она упрощает развёртывание и изоляцию зависимостей.
"""

chunks = text.strip().split("\n\n")

print(chunks)
print(len(chunks))  # 3

['FastAPI — это современный веб-фреймворк для построения API на Python.\nОн автоматически генерирует документацию и поддерживает асинхронность.', 'Alembic — это инструмент для управления миграциями базы данных в SQLAlchemy.\nОн позволяет безопасно обновлять схему БД при изменении моделей.', 'Docker — это платформа для контейнеризации приложений.\nОна упрощает развёртывание и изоляцию зависимостей.']
3


#### Шаг 3. Настройка embedding-функции

##### Chroma может автоматически генерировать эмбеддинги, если вы укажете embedding-функцию:
* Одна и та же модель должна использоваться и для документов, и для запросов — это критично для точности поиска.

In [4]:
from chromadb.utils import embedding_functions

ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

/opt/anaconda3/envs/LLM_course/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8263.14it/s]


#### Шаг 4. Сохранение в Chroma

Библиотека **ChromaDB** реализует несколько типов клиентов для работы с векторным хранилищем.
Наиболее важные из них (в рамках курса) — Client, EphemeralClient и PersistentClient.

* EphemeralClient() — создаёт **временное хранилище в оперативной памяти**. Все данные удаляются после завершения программы. Подходит для тестов и быстрого прототипирования.
* PersistentClient(path="./chroma_db") — сохраняет данные **на диск** в указанную директорию. Коллекции и документы остаются доступны между запусками скрипта.
* Client() — универсальный клиент, который **может работать как ephemeral, так и persistent**, в зависимости от переданного объекта Settings.
Например, так можно заставить Client вести себя как PersistentClient:

**Без Settings (или с настройками по умолчанию) Client() ведёт себя как EphemeralClient**

---
Мы же будем использовать более явный  EphemeralClient

In [6]:
import chromadb

# Создаём in-memory клиент
client = chromadb.EphemeralClient()

# Создаём коллекцию с embedding-функцией
collection = client.create_collection(
    name="docs",
    embedding_function=ef
)

# Добавляем документы — embeddings генерируются автоматически
collection.add(
    ids=[f"id_{i}" for i in range(len(chunks))],
    documents=chunks
)

#### Шаг 5. Семантический поиск

In [8]:
results = collection.query(
    query_texts=["Как работать с миграциями в Python?"],
    n_results=2
)

print(results["documents"][0][0])
# → Alembic — это инструмент для управления миграциями базы данных...

FastAPI — это современный веб-фреймворк для построения API на Python.
Он автоматически генерирует документацию и поддерживает асинхронность.


#### Шаг 6. Связь с RAG

Этот результат — основа **RAG (Retrieval-Augmented Generation):**

1) Пользователь задаёт вопрос.
2) Chroma находит релевантные фрагменты.
3) Фрагменты передаются в LLM как контекст:

```
SYSTEM: Отвечай только на основе предоставленного контекста.
CONTEXT: Alembic используется для миграций БД.
USER: Как обновить схему БД?
```

                  
4) LLM генерирует ответ, **опираясь только на ваши данные.**

#### Примечание

* **In-memory** режим (chromadb.Client()или chromadb.EphemeralClient()) — **только для обучения.** Данные исчезнут после завершения скрипта.
* Для сохранения между запусками используйте:

```
client = chromadb.PersistentClient(path="./my_db")
```

* **Оптимальный размер чанка:** 300–500 токенов — достаточно для смысла, не слишком длинный для LLM.

---

In [13]:
client.clear_system_cache()

In [14]:
from chromadb.utils import embedding_functions
import chromadb

documents = [
    "FastAPI — это современный фреймворк для построения API на Python.",
    "Alembic используется для управления миграциями базы данных в SQLAlchemy.",
    "Docker упрощает развёртывание приложений через контейнеризацию."
]

query = "Как обновить структуру базы данных?"

# Настройка Chroma
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
client = chromadb.Client()
collection = client.create_collection(name="docs", embedding_function=ef)

# Добавление документов
collection.add(
    ids=[f"id_{i}" for i in range(len(documents))],
    documents=documents
)

# Поиск
results = collection.query(query_texts=[query], n_results=1)

# Вывод ТОЛЬКО текста (без лишних слов)
print(results["documents"][0][0])

Alembic используется для управления миграциями базы данных в SQLAlchemy.


In [22]:
def chunk_text(text: str) -> list[str]:
    return [chunk.strip() for chunk in text.split("\n\n") if chunk!='']

In [23]:
chunk_text("A\n\nB\n\n")

['A', 'B']

In [38]:
def is_relevant(query: str, retrieved_chunk: str) -> bool:
    query_list = [x for x in query.lower().split() if len(x)>=3]
    retrieved_chunk_list = [x for x in retrieved_chunk.lower().split() if len(x)>=3]
    
    return any([query in retrieved_chunk.lower() for query in query_list])

In [39]:
is_relevant("Как работать с миграциями", "Alembic используется для управления миграциями.")

True